# DNAKit 核心端到端工作流

本 Notebook 调用稳定的核心公开 API，覆盖 I/O、描述符、指纹、相似度、去重、划分和确定性 SVG 可视化。所有文件输出都写入临时目录；高级流程见 `01_advanced_workflow.ipynb`。

In [ ]:
import sys
import tempfile
from io import StringIO
from pathlib import Path

import dnakit
from dnakit import read_set, write
from dnakit.datasets import SplitConfig, deduplicate, split
from dnakit.descriptors import gc_at_content
from dnakit.fingerprints import kmer
from dnakit.similarity import fingerprint_similarity, similarity_matrix
from dnakit.visualization import (
    Highlight,
    plot_sequence,
    plot_similarity_matrix,
    save_svg,
)

assert sys.version_info >= (3, 10)
assert dnakit.__version__ == "0.1.3"
print(f"DNAKit {dnakit.__version__} on Python {sys.version.split()[0]}")

## 1. 读取固定 FASTA，并用结构化 JSON 往返

`read_set()` 会显式物化全部记录；大文件应使用 `read()` 的单次消费 `RecordSource`。

In [ ]:
fasta = StringIO(
    ">seq-a fixed example\nACGTACGT\n"
    ">seq-b exact duplicate\nACGTACGT\n"
    ">seq-c one substitution\nACGTTCGT\n"
)
records = read_set(fasta, format="fasta")
assert records.ids == ("seq-a", "seq-b", "seq-c")

json_buffer = StringIO()
write_result = write(records, json_buffer, format="json")
round_trip = read_set(StringIO(json_buffer.getvalue()), format="json")
assert round_trip == records
assert write_result.record_count == 3
print({"record_ids": records.ids, "json_records": write_result.record_count})

## 2. 描述符、指纹与相似度

这里计算全序列描述符，并比较两条序列的基础指纹。

In [ ]:
gc_by_id = {record.id: gc_at_content(record).gc_fraction for record in records}
left_fp = kmer(records[0], k=2, mode="binary")
right_fp = kmer(records[2], k=2, mode="binary")
fp_similarity = fingerprint_similarity(left_fp, right_fp, metric="jaccard")

assert gc_by_id == {"seq-a": 0.5, "seq-b": 0.5, "seq-c": 0.5}
assert 0.0 <= fp_similarity.value <= 1.0
print({"gc": gc_by_id, "fingerprint_jaccard": fp_similarity.value})

## 3. 精确去重与可复现划分

去重结果保存分组审计；划分结果保存 seed、shuffle 和分配策略。

In [ ]:
nonredundant = deduplicate(records, equivalence="exact")
partitions = split(
    nonredundant.records,
    config=SplitConfig(
        method="random",
        ratios={"train": 0.5, "test": 0.5},
        seed=7,
    ),
)

assert nonredundant.output_count == 2
assert dict(partitions.counts) == {"train": 1, "test": 1}
assert partitions.seed == 7
print({"deduplicated": nonredundant.output_count, "split_counts": dict(partitions.counts)})

## 4. 生成并安全保存 SVG

序列、高亮和相似度矩阵都生成独立 SVG。`save_svg()` 默认拒绝覆盖并返回 SHA-256 artifact 审计。

In [ ]:
sequence_svg = plot_sequence(
    records[2],
    highlights=[Highlight(4, 5, label="substitution")],
)
matrix_result = similarity_matrix(
    nonredundant.records,
    method="kmer_jaccard",
    k=2,
)
matrix_svg = plot_similarity_matrix(matrix_result)

with tempfile.TemporaryDirectory(prefix="dnakit-notebook-") as directory:
    output_dir = Path(directory)
    artifacts = {
        "sequence": sequence_svg,
        "similarity_matrix": matrix_svg,
    }
    saved = {
        name: save_svg(artifact, output_dir / f"{name}.svg")
        for name, artifact in artifacts.items()
    }
    for name, artifact in artifacts.items():
        assert (output_dir / f"{name}.svg").read_text(encoding="utf-8") == artifact.svg
        assert saved[name].source_sha256 == artifact.sha256

print({name: artifact.sha256[:12] for name, artifact in artifacts.items()})

## 核心流程边界

本 Notebook 为最小核心流程，只保存 SVG。完整项目另提供可选 PNG/TIFF/PDF、热力学、近似聚类、多约束划分、综合评价和分子生物学 API。全相似度矩阵仍为 `O(n²)` 并受 `max_items` 约束，Levenshtein 动态规划受 `max_cells` 约束。